In [4]:
import sys
!{sys.executable} -m pip install lifelines
!{sys.executable} -m pip install jinja2


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jinja2]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import pandas as pd
from lifelines import CoxPHFitter

df = pd.read_csv('cleaned_tmz_94_cohort.csv')

df['is_male'] = (df['gender.demographic'] == 'male').astype(int)

cols_to_use = ['time', 'event', 'cg00000029', 'age_at_index.demographic', 'is_male']
data_final = df[cols_to_use].dropna()

lasso = CoxPHFitter(penalizer=0.1, l1_ratio=1.0)
lasso.fit(data_final, duration_col='time', event_col='event')

print("--- LASSO RESULTS ---")
lasso.print_summary()

enet = CoxPHFitter(penalizer=0.1, l1_ratio=0.5)
enet.fit(data_final, duration_col='time', event_col='event')

print("\n--- ELASTIC NET RESULTS ---")
enet.print_summary()

lasso.summary.to_csv('lasso_results.csv')

enet.summary.to_csv('enet_results.csv')

--- LASSO RESULTS ---


<lifelines.CoxPHFitter: fitted with 94 total observations, 33 right-censored observations>
             duration col = 'time'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 1.0
      baseline estimation = breslow
   number of observations = 94
number of events observed = 61
   partial log-likelihood = -212.81
         time fit was run = 2026-03-20 04:21:10 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
cg00000029               -0.00      1.00      0.00           -0.01            0.01                0.99                1.01
age_at_index.demographic  0.03      1.03      0.01            0.01            0.05                1.01                1.05
is_male                   0.10      1.10      0.27           -0.43            0.62                0.65                1.86

                          cmp to     z    p  -log2(p)
covariate                                            
cg00000029                  0.00 -0.00 1.00      0.00
age_at_index.demographic    0.00  2.43 0.01      6.06
is_male                     0.00  0.36 0.72      0.48
---
Concordance = 0.67
Partial AIC = 431.62
log-likelihood ratio test = 6.52 on 3 df
-log2(p) of ll-ratio test = 3.49


--- ELASTIC NET RESULTS ---


<lifelines.CoxPHFitter: fitted with 94 total observations, 33 right-censored observations>
             duration col = 'time'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.5
      baseline estimation = breslow
   number of observations = 94
number of events observed = 61
   partial log-likelihood = -211.01
         time fit was run = 2026-03-20 04:21:10 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
cg00000029               -0.00      1.00      0.01           -0.01            0.01                0.99                1.01
age_at_index.demographic  0.03      1.03      0.01            0.01            0.05                1.01                1.06
is_male                   0.24      1.28      0.26           -0.26            0.75                0.77                2.13

                          cmp to     z      p  -log2(p)
covariate                                              
cg00000029                  0.00 -0.00   1.00      0.00
age_at_index.demographic    0.00  2.92 <0.005      8.17
is_male                     0.00  0.94   0.35      1.53
---
Concordance = 0.67
Partial AIC = 428.02
log-likelihood ratio test = 10.12 on 3 df
-log2(p) of ll-ratio test = 5.83

In [ ]:
lasso_tuned = CoxPHFitter(penalizer=0.01, l1_ratio=1.0)
lasso_tuned.fit(data_final, duration_col='time', event_col='event')

print("--- TUNED LASSO RESULTS ---")
lasso_tuned.print_summary()

lasso.summary.to_csv('tuned_lasso_results.csv')


--- TUNED LASSO RESULTS ---


<lifelines.CoxPHFitter: fitted with 94 total observations, 33 right-censored observations>
             duration col = 'time'
                event col = 'event'
                penalizer = 0.01
                 l1 ratio = 1.0
      baseline estimation = breslow
   number of observations = 94
number of events observed = 61
   partial log-likelihood = -208.10
         time fit was run = 2026-03-20 04:23:35 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
cg00000029               -0.23      0.79      0.74           -1.69            1.22                0.18                3.40
age_at_index.demographic  0.04      1.04      0.01            0.02            0.07                1.02                1.07
is_male                   0.40      1.49      0.28           -0.14            0.94                0.87                2.57

                          cmp to     z      p  -log2(p)
covariate                                              
cg00000029                  0.00 -0.31   0.75      0.41
age_at_index.demographic    0.00  3.51 <0.005     11.10
is_male                     0.00  1.44   0.15      2.75
---
Concordance = 0.67
Partial AIC = 422.19
log-likelihood ratio test = 15.95 on 3 df
-log2(p) of ll-ratio test = 9.75